# Off-the-shelf Vision Model Testing — Kvasir-Capsule

Tests 4 models on capsule endoscopy images:
1. **CLIP** — zero-shot classification
2. **BLIP-2** — visual question answering
4. **LLaVA 1.5 7B (8-bit)** — vision language model

**Runtime: GPU required** → Runtime → Change runtime type → T4 GPU

In [ ]:
!pip install -q transformers accelerate bitsandbytes Pillow einops

In [ ]:
from google.colab import files
import tarfile, os, random
from pathlib import Path

print('Upload your 4 tar.gz files: ulcer.tar.gz, erosion.tar.gz, blood_fresh.tar.gz, polyp.tar.gz')
uploaded = files.upload()

os.makedirs('/content/samples', exist_ok=True)

for fname in uploaded.keys():
    print(f'Extracting {fname}...')
    with tarfile.open(fname, 'r:gz') as t:
        t.extractall('/content/samples')
print()
for item in sorted(Path('/content/samples').iterdir()):
    if item.is_dir():
        imgs = list(item.glob('*.jpg')) + list(item.glob('*.png'))
        print(f'  {item.name}/: {len(imgs)} images')

Upload your 4 tar.gz files: ulcer.tar.gz, erosion.tar.gz, blood_fresh.tar.gz, polyp.tar.gz


Saving blood_fresh.tar.gz to blood_fresh.tar.gz
Saving erosion.tar.gz to erosion.tar.gz
Saving polyp.tar.gz to polyp.tar.gz
Saving ulcer.tar.gz to ulcer.tar.gz
Extracting blood_fresh.tar.gz...
Extracting erosion.tar.gz...


/tmp/ipykernel_2860/2583352260.py:14: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  t.extractall('/content/samples')


Extracting polyp.tar.gz...
Extracting ulcer.tar.gz...

  Blood - fresh/: 446 images
  Erosion/: 506 images
  Polyp/: 55 images
  Ulcer/: 854 images


In [ ]:
from pathlib import Path

base = Path('/content/samples')
print('Contents of /content/samples:')
for item in sorted(base.rglob('*'))[:30]:
    print(f'  {item}')

In [1]:
import torch
import json
import random
from pathlib import Path
from PIL import Image
CLASS_FOLDERS = {
    'ulcer':       'Ulcer',
    'erosion':     'Erosion',
    'blood fresh': 'Blood - fresh',
    'polyp':       'Polyp',
}
CLASSES          = list(CLASS_FOLDERS.keys())
BASE_DIR         = Path('/content/samples')
OUTPUT_DIR       = Path('/content/results')
SAMPLES_PER_CLASS = 10
DEVICE           = 'cuda' if torch.cuda.is_available() else 'cpu'

OUTPUT_DIR.mkdir(exist_ok=True)

print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

# Sample images
random.seed(42)
samples = {}
for cls, folder in CLASS_FOLDERS.items():
    path = BASE_DIR / folder
    if not path.exists():
        # Try alternate names
        for alt in [cls, cls.replace(' ', '_'), cls.replace(' ', '-')]:
            if (BASE_DIR / alt).exists():
                path = BASE_DIR / alt
                break
    imgs = list(path.glob('*.jpg')) + list(path.glob('*.png'))
    samples[cls] = random.sample(imgs, min(SAMPLES_PER_CLASS, len(imgs)))
    print(f'  {cls}: {len(samples[cls])} images sampled')


def save_results(results, name):
    path = OUTPUT_DIR / f'{name}.json'
    with open(path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f'Saved: {path}')


def print_summary(results, model_name):
    print(f'\n{"="*50}')
    print(f' {model_name} — Summary')
    print(f'{"="*50}')
    correct = sum(1 for r in results if r.get('correct'))
    total   = len(results)
    print(f'Accuracy: {correct}/{total} = {correct/total:.1%}')
    for cls in CLASSES:
        cls_r = [r for r in results if r['true_class'] == cls]
        cls_c = sum(1 for r in cls_r if r.get('correct'))
        print(f'  {cls:<15}: {cls_c}/{len(cls_r)}')
    print('='*50)

Device : cuda
GPU    : Tesla T4
VRAM   : 14.6 GB
  ulcer: 0 images sampled
  erosion: 0 images sampled
  blood fresh: 0 images sampled
  polyp: 0 images sampled


In [ ]:
import os
for item in os.listdir('/content/samples'):
    print(repr(item))

In [ ]:
# Cell 4 — CLIP (zero-shot classification)
from transformers import CLIPModel, CLIPProcessor

print('Loading CLIP...')
clip_model     = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(DEVICE)
clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')

text_prompts = [
    'a capsule endoscopy image showing an ulcer',
    'a capsule endoscopy image showing erosion',
    'a capsule endoscopy image showing fresh blood',
    'a capsule endoscopy image showing a polyp',
]

clip_results = []
for true_cls in CLASSES:
    for img_path in samples[true_cls]:
        image  = Image.open(img_path).convert('RGB')
        inputs = clip_processor(text=text_prompts, images=image,
                                return_tensors='pt', padding=True).to(DEVICE)
        with torch.no_grad():
            probs = clip_model(**inputs).logits_per_image.softmax(dim=1).squeeze().tolist()

        pred_cls = CLASSES[probs.index(max(probs))]
        clip_results.append({
            'model':      'CLIP',
            'image':      img_path.name,
            'true_class': true_cls,
            'pred_class': pred_cls,
            'correct':    pred_cls == true_cls,
            'confidence': round(max(probs), 4),
            'all_probs':  {CLASSES[i]: round(p, 4) for i, p in enumerate(probs)}
        })

save_results(clip_results, 'clip_results')
print_summary(clip_results, 'CLIP')

del clip_model, clip_processor
torch.cuda.empty_cache()

In [ ]:
# Cell 5 — BLIP-2 (visual question answering)
from transformers import BlipProcessor, BlipForQuestionAnswering

QUESTIONS = [
    'What abnormality is visible in this endoscopy image?',
    'Is there any bleeding visible?',
    'What type of lesion is shown?',
    'Describe what you see in this capsule endoscopy frame.',
]

print('Loading BLIP-2...')
blip_processor = BlipProcessor.from_pretrained('Salesforce/blip-vqa-base')
blip_model     = BlipForQuestionAnswering.from_pretrained(
    'Salesforce/blip-vqa-base', torch_dtype=torch.float16
).to(DEVICE)

blip_results = []
for true_cls in CLASSES:
    for img_path in samples[true_cls]:
        image   = Image.open(img_path).convert('RGB')
        answers = {}
        for q in QUESTIONS:
            inputs = blip_processor(image, q, return_tensors='pt').to(DEVICE)
            with torch.no_grad():
                out = blip_model.generate(**inputs)
            answers[q] = blip_processor.decode(out[0], skip_special_tokens=True)

        blip_results.append({
            'model':      'BLIP-2',
            'image':      img_path.name,
            'true_class': true_cls,
            'answers':    answers
        })

save_results(blip_results, 'blip2_results')

print('\nSample BLIP-2 outputs:')
for r in blip_results[:3]:
    print(f"\n  [{r['true_class']}] {r['image']}")
    for q, a in r['answers'].items():
        print(f'    Q: {q}')
        print(f'    A: {a}')

del blip_model, blip_processor
torch.cuda.empty_cache()

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1

In [ ]:
# Cell 7 — LLaVA 1.5 7B (8-bit quantized)
from transformers import LlavaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

print('Loading LLaVA 1.5 7B (8-bit)... takes ~2 minutes')
quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True # Enable CPU offload for 8-bit quantization
)

llava_model = LlavaForConditionalGeneration.from_pretrained(
    'llava-hf/llava-1.5-7b-hf',
    quantization_config=quant_config,
    device_map='auto',
)
llava_processor = AutoProcessor.from_pretrained('llava-hf/llava-1.5-7b-hf')

print(f'Model loaded. VRAM used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB')

def ask_llava(image, question):
    prompt  = f'USER: <image>\n{question}\nASSISTANT:'
    inputs  = llava_processor(text=prompt, images=image, return_tensors='pt').to('cuda')
    with torch.no_grad():
        output = llava_model.generate(**inputs, max_new_tokens=100, do_sample=False)
    full = llava_processor.decode(output[0], skip_special_tokens=True)
    return full.split('ASSISTANT:')[-1].strip()

llava_results = []
for true_cls in CLASSES:
    for img_path in samples[true_cls]:
        image   = Image.open(img_path).convert('RGB')
        answers = {q: ask_llava(image, q) for q in QUESTIONS}
        llava_results.append({
            'model':      'LLaVA-1.5-7B-8bit',
            'image':      img_path.name,
            'true_class': true_cls,
            'answers':    answers
        })

save_results(llava_results, 'llava_results')

print('\nSample LLaVA outputs:')
for r in llava_results[:3]:
    print(f"\n  [{r['true_class']}] {r['image']}")
    for q, a in r['answers'].items():
        print(f'    Q: {q}')
        print(f'    A: {a[:150]}')

del llava_model, llava_processor
torch.cuda.empty_cache()

In [ ]:
import shutil

shutil.make_archive('/content/all_results', 'zip', '/content/results')
files.download('/content/all_results.zip')
print('Downloaded all_results.zip')